# 第15讲：从调查经历提炼 Skill

本练习对应复杂 On-call 调查后的技能提炼设计题。先思考：复盘应找哪些学习信号？怎样组织步骤、工具使用和判断条件？什么经验应剔除？怎样确定适用范围、避免重复与碎片化？没有可靠流程时可以不创建。

这里用连接泄漏构造一段可检查的调查，主动触发原生后台复盘，再阅读文件变化；不是对设计题自动给分。


In [ ]:
from pathlib import Path
import ast
import json
import os
import signal
import subprocess
from IPython.display import Code, display

if "process" in globals() and process is not None:
    stop_owned()
HERE = Path.cwd().resolve()
if not (HERE / "native_workshop.py").exists():
    HERE = HERE / "examples/15-skill-autocreation"
assert (HERE / "native_workshop.py").is_file()
REPO = HERE.parents[1]
HERMES_SRC = Path(os.environ.get("HERMES_SRC", REPO / ".deps/hermes-agent")).resolve()
HERMES_PYTHON = Path(os.environ.get("HERMES_PYTHON", HERMES_SRC / ".venv/bin/python"))
assert HERMES_PYTHON.is_file(), "请先运行仓库 scripts/setup_deps.sh"
env = dict(os.environ, HERMES_SRC=str(HERMES_SRC), PYTHONDONTWRITEBYTECODE="1",
           PYTHONPATH=str(HERMES_SRC) + os.pathsep + os.environ.get("PYTHONPATH", ""))
process = None
live_result = None

def stop_owned():
    global process
    if process is not None and process.poll() is None:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait(timeout=10)
    captured = ""
    if process is not None and process.stdout is not None:
        try:
            captured, _ = process.communicate(timeout=1)
        except (ValueError, subprocess.TimeoutExpired):
            pass
        process.stdout.close()
    process = None
    return captured


## 先看实际执行代码

下面只展示将要执行的源码，不请求模型。`foreground()` 内的 `AIAgent.run_conversation()` 用空历史启动真实调查；`review_job()` 中的 `spawn_background_review_thread()` 创建原生复盘，观察器只记录返回和回执。


In [ ]:
source = (HERE / "native_workshop.py").read_text()
tree = ast.parse(source)
for node in tree.body:
    if isinstance(node, ast.FunctionDef) and node.name in ("foreground", "review_job"):
        display(Code(ast.get_source_segment(source, node), language="python"))


## 检查构造现场与安全反例

不需要密钥。`test` 工具实际执行正常和异常分支；构造服务 alpha 在抛出异常时漏归还，beta 在提前返回时漏归还。这是业务重放，不是历史模型回放。自检全部使用明确构造的调用，不能作为模型学习结果。


In [ ]:
for script in ("native_workshop.py", "selfcheck.py"):
    checked = subprocess.run([str(HERMES_PYTHON), "-B", str(HERE / script)],
                             cwd=HERE, env=env, text=True, capture_output=True)
    print(checked.stdout)
    print(checked.stderr)
    checked.check_returncode()


## 完成调查并主动触发复盘

设置当前进程的 `GLM_API_KEY`，然后把 `RUN_MODEL` 改为 `True`。默认模型为 `glm-5.2`、官方接口，可用环境变量覆盖。下面启动的脚本先完成调查，再显式触发原生复盘；自动Skill阈值已关闭，所以这不是Turn收尾自动触发的验证。

复盘会先检查已有方法，再决定是否创建。脚本不提供新方法正文；没有更新时保留原生判断，不强行制造文件。调用会产生API费用。


In [ ]:
from uuid import uuid4
import sys
sys.path.insert(0, str(HERE.parent))
from notebook_native_results import collect_run, save_observation
from common import scrub
RUN_MODEL = os.environ.get("LESSON15_RUN_MODEL") == "1"  # 交互运行时可改为 True
live_result = None
if not RUN_MODEL:
    print("尚未运行模型；上面的现场检查不代表 Skill 学习成功。")
else:
    for name in ("GLM_API_KEY", "GLM_MODEL", "GLM_BASE_URL"):
        if name in os.environ:
            env[name] = os.environ[name]
    current_run = Path(os.environ.get("LESSON15_RUN_DIR") or (HERE / "runtime" / ("native-notebook-" + uuid4().hex)))
    if current_run.exists() or current_run.is_symlink():
        raise ValueError("本次运行目录已存在，请使用新的运行目录，不读取历史记录。")
    validation_id = os.environ.get("LESSON15_VALIDATION_ID")
    child, output, process_error = None, "", None
    try:
        process = child = subprocess.Popen(
            [str(HERMES_PYTHON), "-B", str(HERE / "native_workshop.py"), "--live",
             "--run-dir", str(current_run)],
            cwd=HERE, env=env, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, start_new_session=True)
        output, _ = child.communicate(timeout=900)
    except (Exception, KeyboardInterrupt) as error:
        process_error = type(error).__name__ + ": " + str(error)
        partial = getattr(error, "output", "") or ""
        output = partial.decode(errors="replace") if isinstance(partial, bytes) else partial
    finally:
        captured = stop_owned()
        if captured:
            output = captured
    if current_run.is_dir():
        (current_run / "subprocess.log").write_text(scrub(output))
    print(scrub(output))
    live_result = collect_run(current_run, validation_id,
                              child.returncode if child else None, process_error)
    save_observation(HERE, live_result)
    print("本次状态：", live_result["status"], "；继续查看本轮已有记录。")


## 看原生回执、文件变化与后续使用

下面只读取为刚才进程分配的独立运行目录，不寻找历史结果。进程非零退出、超时或复盘失败时，仍展示本轮已有调查、原始复盘回执及错误；尚未进入的阶段明确显示未取得。检查主文件是否以步骤为主、教训是否嵌入条件、一次性信息是否剔除，以及工具细节是否进入引用文件；这些内容判断由你完成。没有创建时，阅读后台说明及查库记录。


In [ ]:
from notebook_native_results import show_observation
if live_result is None:
    print("尚无本次模型运行，不展示历史 Skill 充当本次结果。")
else:
    show_observation(live_result)


## 收尾与验证范围

运行记录位于独立 `runtime/native-*/`，不提交Git。正常结束、异常或中断都处理本格自己启动的进程，不停止其他运行。当前版本尚未完成模型端到端验证；源码没有预置执行输出。

新Session复用是扩展观察，不是要求每次复盘必须创建Skill。固定Hermes版本实际只有 `_SKILL_REVIEW_PROMPT`，不依赖正文提及但该版本不存在的另外两个提示块变量。


In [ ]:
stop_owned()
